# Phase 11: Supervised and Semi-Supervised Cervical Cell Classification

**Fixed backbone:** ConvNeXt-Tiny (ImageNet pretrained)  
**Datasets:** CRIC and RIVA  
**Label budgets:** 5% and 10% of each dataset's fixed training split  
**Methods:** supervised baseline, pseudo-labeling, Mean Teacher, FixMatch  
**Primary checkpoint-selection metric:** validation Macro-F1  
**Primary reporting:** accuracy, precision, sensitivity/recall, specificity, Macro-F1, AUROC, and confusion-matrix counts.

## Experimental protocol and parity

- Reuse the existing `data/splits_v2/cric_splits_v2.csv` and `riva_splits_v2.csv`. This notebook does not create or modify train/validation/test splits.
- The labeled subset is sampled deterministically and class-stratified from the **training split only**. Its CSV is saved and reused across methods at the same dataset and label fraction. The remaining training rows form the unlabeled pool for semi-supervised methods.
- Validation is used for early stopping and best-checkpoint selection. Test data are not used for training or checkpoint selection.
- Both datasets use the same backbone, input size, optimizer, learning rate, weight decay, batch size, epoch limit, patience, normalization, and method hyperparameters. This is intended to preserve experimental parity rather than tune settings separately for each dataset.
- Preprocessing is fixed at 128×128 RGB with ImageNet normalization. Training uses the same mild geometric augmentation for all runs. FixMatch's strong view adds mild color perturbation.
- The run queue contains 2 datasets × 2 label fractions × 4 methods = **16 experiments**. Each run has its own output directory, checkpoints, history CSV, and metrics JSON.
- **Restart behavior:** the latest checkpoint is saved after each completed training batch. It includes model/optimizer state, RNG states, epoch/batch cursor, best validation score, patience counter, and (for Mean Teacher) teacher weights. On a later Run All, completed runs are skipped and interrupted runs resume from their last saved batch checkpoint. If power fails during a checkpoint write, the previous checkpoint remains available.
- Run All can take a long time. Interrupt the kernel to stop; do not delete the output directory if you want to resume. Keep the configuration unchanged when resuming.

## Important assumptions

The split manifests are expected to include `crop_path`, `binary_label`, and `split`, as used in the project’s V2 pipeline. The notebook checks these columns and reports errors rather than silently inventing mappings. Paths are resolved relative to the repository root when necessary. `binary_label=0` is treated as normal/non-pathological and `binary_label=1` as abnormal/pathological.

These are controlled baseline implementations of the named semi-supervised methods. They are not claimed to be exact reproductions of every paper-specific variant. Results should be interpreted as experimental evidence, not clinical validation.

In [1]:
# 1. Imports, paths, configuration
from pathlib import Path
import os, json, time, random, hashlib, math, copy, warnings
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

SEED = 42
DATASETS = ("cric", "riva")
FRACTIONS = (0.05, 0.10)
METHODS = ("supervised", "pseudo_label", "mean_teacher", "fixmatch")
IMAGE_SIZE = 128
BATCH_SIZE = 32
MAX_EPOCHS = 100
PATIENCE = 12
LR = 1e-4
WEIGHT_DECAY = 1e-4
PSEUDO_THRESHOLD = 0.95
UNSUP_WEIGHT = 1.0
EMA_DECAY = 0.99
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

# Find repo root, or set ROOT explicitly if your notebook is elsewhere.
_candidates = [Path.cwd(), Path.cwd().parent, Path.home() / "Documents" / "cervical-semi-sl"]
ROOT = next((p.resolve() for p in _candidates if (p / "data" / "splits_v2" / "cric_splits_v2.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Repo root not found. Set ROOT manually to your cervical-semi-sl folder.")
SPLIT_DIR = ROOT / "data" / "splits_v2"
OUT_DIR = ROOT / "data" / "results" / "phase11_controlled_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
weak_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
strong_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=.15, contrast=.15, saturation=.10, hue=.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

CONFIG = {
    "seed": SEED, "datasets": DATASETS, "fractions": FRACTIONS, "methods": METHODS,
    "backbone": "convnext_tiny", "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "lr": LR,
    "weight_decay": WEIGHT_DECAY, "pseudo_threshold": PSEUDO_THRESHOLD,
    "unsup_weight": UNSUP_WEIGHT, "ema_decay": EMA_DECAY,
}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]

print("ROOT:", ROOT)
print("Output:", OUT_DIR)
print("Device:", DEVICE, "| AMP:", AMP)
print("Configuration hash:", CONFIG_HASH)


ROOT: C:\Users\nanda\Documents\cervical-semi-sl
Output: C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1
Device: cuda | AMP: True
Configuration hash: c995dc2d1ab462b3


## 2. Load and audit the fixed split manifests

This cell validates the expected schema, split names, binary labels, duplicates, crop existence, and class counts. If it fails, fix the manifest/path issue before starting training. Do not regenerate splits as a workaround.

In [2]:
def resolve_crop(p):
    p = Path(str(p))
    candidates = [p] if p.is_absolute() else [ROOT / p, ROOT / "data" / p]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return None

manifests = {}
for ds in DATASETS:
    csv_path = SPLIT_DIR / f"{ds}_splits_v2.csv"
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)
    required = {"crop_path", "binary_label", "split"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{ds.upper()} missing required columns {missing}. Actual columns: {list(df.columns)}")
    df = df.copy()
    df["split"] = df["split"].astype(str).str.strip().str.lower().replace({"validation":"val", "valid":"val"})
    if set(df["split"].unique()) != {"train", "val", "test"}:
        raise ValueError(f"{ds.upper()} split values are {sorted(df['split'].unique())}, expected train/val/test.")
    df["binary_label"] = pd.to_numeric(df["binary_label"], errors="raise").astype(int)
    if not set(df["binary_label"].unique()).issubset({0, 1}):
        raise ValueError(f"{ds.upper()} labels must be 0/1.")
    df["resolved_path"] = df["crop_path"].map(resolve_crop)
    if df["resolved_path"].isna().any():
        examples = df.loc[df.resolved_path.isna(), "crop_path"].head(5).tolist()
        raise FileNotFoundError(f"{ds.upper()}: crop paths unresolved. Examples: {examples}")
    if df["resolved_path"].duplicated().any():
        raise ValueError(f"{ds.upper()}: duplicate resolved crop paths. Verify the split manifest key.")
    manifests[ds] = df.reset_index(drop=True)
    print(f"\n{ds.upper()} rows={len(df):,}")
    print(pd.crosstab(df["split"], df["binary_label"], margins=True))
    with Image.open(df.iloc[0]["resolved_path"]) as im:
        print("Sample crop:", im.size, im.mode)



CRIC rows=11,534
binary_label     0     1    All
split                          
test          1603   704   2307
train         4482  3592   8074
val            694   459   1153
All           6779  4755  11534
Sample crop: (128, 128) RGB

RIVA rows=15,949
binary_label      0     1    All
split                           
test           2070  1126   3196
train          7332  3839  11171
val            1115   467   1582
All           10517  5432  15949
Sample crop: (128, 128) RGB


## 3. Deterministic labeled subsets and datasets

A separate labeled subset is created for each dataset/fraction, with both classes represented and sampled proportionally within each class. It is saved under `phase11_controlled_v1/label_subsets/` and reused on subsequent sessions. The selected rows are drawn only from `train`. All other training rows become unlabeled data for the semi-supervised methods.

In [3]:
def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def labeled_frames(ds, frac):
    train = manifests[ds].query("split == 'train'").reset_index(drop=True)
    subset_dir = OUT_DIR / "label_subsets"
    subset_dir.mkdir(parents=True, exist_ok=True)
    subset_path = subset_dir / f"{ds}_labeled_{int(frac*100)}pct_seed{SEED}.csv"
    if subset_path.exists():
        saved = pd.read_csv(subset_path)
        wanted = set(saved["resolved_path"].astype(str))
        lab = train[train.resolved_path.astype(str).isin(wanted)].copy()
        if len(lab) != len(wanted):
            raise ValueError(f"Saved subset mismatch: {subset_path}. Do not silently resample.")
    else:
        rng = np.random.default_rng(SEED + (0 if ds == "cric" else 10000) + int(frac*100))
        picked = []
        for cls in (0, 1):
            idx = np.flatnonzero(train.binary_label.to_numpy() == cls)
            n = max(1, int(round(frac * len(idx))))
            picked.extend(rng.choice(idx, size=min(n, len(idx)), replace=False).tolist())
        lab = train.iloc[sorted(picked)].copy()
        lab[["resolved_path", "binary_label"]].to_csv(subset_path, index=False)
    unlab = train[~train.resolved_path.isin(lab.resolved_path)].copy()
    return lab.reset_index(drop=True), unlab.reset_index(drop=True)

class CellDataset(Dataset):
    def __init__(self, frame, mode):
        self.frame = frame.reset_index(drop=True)
        self.mode = mode
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        with Image.open(r.resolved_path) as im:
            im = im.convert("RGB")
            if self.mode == "unlabeled":
                return weak_tf(im), strong_tf(im)
            tfm = train_tf if self.mode == "train" else weak_tf
            return tfm(im), int(r.binary_label)

def loader(frame, mode, shuffle=False, batch_size=BATCH_SIZE):
    gen = torch.Generator().manual_seed(SEED)
    return DataLoader(CellDataset(frame, mode), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=AMP, drop_last=False, generator=gen)

for ds in DATASETS:
    for frac in FRACTIONS:
        lab, unlab = labeled_frames(ds, frac)
        print(f"{ds.upper()} {int(frac*100)}%: labeled={len(lab):,}, unlabeled={len(unlab):,}")


CRIC 5%: labeled=404, unlabeled=7,670
CRIC 10%: labeled=807, unlabeled=7,267
RIVA 5%: labeled=559, unlabeled=10,612
RIVA 10%: labeled=1,117, unlabeled=10,054


## 4. Model, evaluation metrics, and checkpoint utilities

ConvNeXt-Tiny is ImageNet-pretrained, with a new two-class head. If pretrained weights are not cached, torchvision may need internet access to download them. The checkpoint includes configuration hash validation to prevent accidental resumption with changed hyperparameters.

In [4]:
def build_model():
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, 2)
    return model.to(DEVICE)

def calc_metrics(y, prob):
    y = np.asarray(y, dtype=int); prob = np.asarray(prob, dtype=float)
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0,1]).ravel()
    try: auc = float(roc_auc_score(y, prob))
    except ValueError: auc = float("nan")
    return {
        "accuracy": float(accuracy_score(y, pred)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall_sensitivity": float(recall_score(y, pred, zero_division=0)),
        "specificity": float(tn/(tn+fp)) if tn+fp else float("nan"),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "auroc": auc, "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); ys=[]; probs=[]
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            logits = model(x)
        probs.extend(torch.softmax(logits.float(), dim=1)[:,1].cpu().numpy())
        ys.extend(y.numpy())
    return calc_metrics(ys, probs)

def save_state(path, model, optimizer, scaler, epoch, batch_cursor, best_score, bad_epochs, history, teacher=None):
    state = {
        "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(), "epoch": epoch, "batch_cursor": batch_cursor,
        "best_score": best_score, "bad_epochs": bad_epochs, "history": history,
        "teacher": teacher.state_dict() if teacher is not None else None,
        "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "config_hash": CONFIG_HASH,
    }
    tmp = path.with_suffix(".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)

def load_state(path, model, optimizer, scaler, teacher=None):
    state = torch.load(path, map_location=DEVICE, weights_only=False)

    if state.get("config_hash") != CONFIG_HASH:
        raise ValueError(
            "Checkpoint config differs from current notebook config. "
            "Use the original settings to resume."
        )

    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scaler.load_state_dict(state["scaler"])

    if teacher is not None and state.get("teacher") is not None:
        teacher.load_state_dict(state["teacher"])

    random.setstate(state["python_rng"])
    np.random.set_state(state["numpy_rng"])

    # RNG states must be CPU ByteTensors, even when training on a GPU.
    torch_rng = state["torch_rng"]
    if not torch.is_tensor(torch_rng):
        torch_rng = torch.as_tensor(torch_rng)
    torch.set_rng_state(torch_rng.detach().to(device="cpu", dtype=torch.uint8))

    if torch.cuda.is_available() and state.get("cuda_rng") is not None:
        cuda_rng = [
            s.detach().to(device="cpu", dtype=torch.uint8)
            if torch.is_tensor(s) else torch.as_tensor(s, dtype=torch.uint8)
            for s in state["cuda_rng"]
        ]
        torch.cuda.set_rng_state_all(cuda_rng)

    return state


## 5. Training engine

**Method definitions used here**
- **Supervised:** cross-entropy on labeled samples only.
- **Pseudo-labeling:** confident weak-view predictions on unlabeled samples are used as pseudo-targets.
- **Mean Teacher:** student predictions are regularized toward an exponential-moving-average teacher's predictions.
- **FixMatch:** confident pseudo-labels from a weak view supervise the corresponding strong view.

The supervised and unlabeled batches are cycled independently. The queue saves a checkpoint after every optimizer step. For exact continuation, batch order is generated deterministically from run ID and epoch; the batch cursor records how many batches were completed. A restart can repeat at most the in-progress step if interruption occurs between the optimizer update and its checkpoint write.

In [5]:
def update_ema(teacher, student, decay=EMA_DECAY):
    with torch.no_grad():
        for tp, sp in zip(teacher.parameters(), student.parameters()):
            tp.data.mul_(decay).add_(sp.data, alpha=1-decay)
        for tb, sb in zip(teacher.buffers(), student.buffers()):
            tb.copy_(sb)

def stable_seed(text):
    return int(hashlib.sha256(text.encode()).hexdigest()[:8], 16)

def run_name(ds, frac, method):
    return f"{ds}_{int(frac*100)}pct_{method}_convnext_tiny_seed{SEED}"

def train_experiment(ds, frac, method):
    name = run_name(ds, frac, method)
    rd = OUT_DIR / name
    rd.mkdir(parents=True, exist_ok=True)
    latest, best = rd/"checkpoint_latest.pt", rd/"checkpoint_best.pt"
    hist_path, metrics_path, done_path = rd/"history.csv", rd/"test_metrics.json", rd/"DONE.json"
    if done_path.exists() and metrics_path.exists():
        print("SKIP completed:", name)
        return json.loads(metrics_path.read_text())

    lab, unlab = labeled_frames(ds, frac)
    val_df = manifests[ds].query("split == 'val'").reset_index(drop=True)
    test_df = manifests[ds].query("split == 'test'").reset_index(drop=True)
    train_dl = loader(lab, "train", shuffle=False)
    val_dl = loader(val_df, "eval")
    test_dl = loader(test_df, "eval")
    unlab_ds = CellDataset(unlab, "unlabeled") if method != "supervised" else None

    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    teacher = None
    if method == "mean_teacher":
        teacher = build_model()
        teacher.load_state_dict(model.state_dict())
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad_(False)

    history=[]; start_epoch=0; start_batch=0; best_score=-1.; bad_epochs=0
    if latest.exists():
        state = load_state(latest, model, optimizer, scaler, teacher)
        start_epoch, start_batch = state["epoch"], state["batch_cursor"]
        best_score, bad_epochs, history = state["best_score"], state["bad_epochs"], state["history"]
        print(f"RESUME {name}: epoch {start_epoch+1}, batch {start_batch}")

    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        # Reproducible permutation. Data augmentation randomness is checkpointed as well.
        gen = torch.Generator().manual_seed(SEED + stable_seed(name) + epoch)
        perm = torch.randperm(len(lab), generator=gen).tolist()
        batches = [perm[i:i+BATCH_SIZE] for i in range(0, len(perm), BATCH_SIZE)]
        skip = start_batch if epoch == start_epoch else 0
        if skip >= len(batches):
            start_batch = 0
            continue
        epoch_loss=0.; seen=0
        for bi in range(skip, len(batches)):
            indices = batches[bi]
            items = [train_dl.dataset[j] for j in indices]
            x = torch.stack([z[0] for z in items]).to(DEVICE)
            y = torch.tensor([z[1] for z in items], dtype=torch.long, device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                logits = model(x)
                loss = F.cross_entropy(logits, y)
                if method != "supervised" and len(unlab) > 0:
                    # Deterministic unlabeled indices per global batch; avoids iterator state ambiguity on resume.
                    n_u = min(BATCH_SIZE, len(unlab))
                    u_start = ((epoch * len(batches) + bi) * BATCH_SIZE) % len(unlab)
                    u_idx = [(u_start+j) % len(unlab) for j in range(n_u)]
                    u_items = [unlab_ds[j] for j in u_idx]
                    u_weak = torch.stack([z[0] for z in u_items]).to(DEVICE)
                    u_strong = torch.stack([z[1] for z in u_items]).to(DEVICE)
                    if method == "pseudo_label":
                        with torch.no_grad():
                            p = torch.softmax(model(u_weak), dim=1)
                            conf, pseudo = p.max(dim=1)
                        mask = conf >= PSEUDO_THRESHOLD
                        if mask.any():
                            loss = loss + UNSUP_WEIGHT * F.cross_entropy(model(u_weak[mask]), pseudo[mask])
                    elif method == "mean_teacher":
                        with torch.no_grad():
                            teacher_p = torch.softmax(teacher(u_weak), dim=1)
                        student_p = torch.softmax(model(u_strong), dim=1)
                        loss = loss + UNSUP_WEIGHT * F.mse_loss(student_p, teacher_p)
                    elif method == "fixmatch":
                        with torch.no_grad():
                            p = torch.softmax(model(u_weak), dim=1)
                            conf, pseudo = p.max(dim=1)
                        mask = conf >= PSEUDO_THRESHOLD
                        if mask.any():
                            loss = loss + UNSUP_WEIGHT * F.cross_entropy(model(u_strong[mask]), pseudo[mask])
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            if teacher is not None: update_ema(teacher, model)
            epoch_loss += float(loss.detach().cpu()) * len(y); seen += len(y)

            # Persist after every completed optimizer step.
            save_state(latest, model, optimizer, scaler, epoch, bi+1, best_score, bad_epochs, history, teacher)

        start_batch=0
        val_metrics = evaluate(model, val_dl)
        row = {"epoch": epoch+1, "train_loss": epoch_loss/max(seen,1),
               **{f"val_{k}":v for k,v in val_metrics.items()}}
        history.append(row)
        score = val_metrics["macro_f1"]
        if score > best_score:
            best_score, bad_epochs = score, 0
            torch.save({"model": model.state_dict(), "epoch": epoch+1,
                        "val_metrics": val_metrics, "config_hash": CONFIG_HASH}, best)
        else:
            bad_epochs += 1
        pd.DataFrame(history).to_csv(hist_path, index=False)
        save_state(latest, model, optimizer, scaler, epoch+1, 0, best_score, bad_epochs, history, teacher)
        print(f"{name} | epoch {epoch+1:03d}/{MAX_EPOCHS} | loss={row['train_loss']:.4f} | "
              f"val_macro_f1={score:.4f} | best={best_score:.4f} | patience={bad_epochs}/{PATIENCE}")
        if bad_epochs >= PATIENCE:
            print("Early stopping:", name)
            break

    if not best.exists():
        raise RuntimeError(f"No best checkpoint was produced for {name}")
    payload = torch.load(best, map_location=DEVICE, weights_only=False)
    model.load_state_dict(payload["model"])
    test_metrics = evaluate(model, test_dl)
    result = {"run_id": name, "dataset": ds, "method": method, "label_fraction": frac,
              "backbone": "convnext_tiny", "seed": SEED, "best_epoch": payload["epoch"],
              "best_val_macro_f1": payload["val_metrics"]["macro_f1"],
              "test_metrics": test_metrics, "config_hash": CONFIG_HASH}
    metrics_path.write_text(json.dumps(result, indent=2))
    done_path.write_text(json.dumps({"status":"COMPLETED", "run_id":name}, indent=2))
    print("TEST:", name, json.dumps(test_metrics, indent=2))
    return result


## 6. Run All: start or resume the complete experiment queue

This is the only cell that starts training. Run All from the top to recreate definitions and then execute the queue. Completed runs are skipped. An interrupted run resumes from its latest batch checkpoint. The summary CSV is refreshed after every completed experiment, so results from earlier runs remain available even if the queue is interrupted.

In [6]:
# RUN ALL EXPERIMENTS
seed_all(SEED)
for ds in DATASETS:
    for frac in FRACTIONS:
        for method in METHODS:
            try:
                train_experiment(ds, frac, method)
            except KeyboardInterrupt:
                print("\\nInterrupted by user. Latest batch checkpoint is preserved. Run All again to resume.")
                raise
            # Rolling summary after each run.
            rows = []
            for p in OUT_DIR.glob("*/test_metrics.json"):
                try:
                    r = json.loads(p.read_text())
                    rows.append({
                        "run_id": r["run_id"], "dataset": r["dataset"], "method": r["method"],
                        "label_fraction": r["label_fraction"], "backbone": r["backbone"],
                        "best_epoch": r["best_epoch"], "best_val_macro_f1": r["best_val_macro_f1"],
                        **r["test_metrics"]
                    })
                except Exception as e:
                    print("Could not parse", p, e)
            if rows:
                pd.DataFrame(rows).sort_values(["dataset","label_fraction","method"]).to_csv(
                    OUT_DIR / "phase11_summary.csv", index=False
                )
print("Queue finished. Summary:", OUT_DIR / "phase11_summary.csv")


SKIP completed: cric_5pct_supervised_convnext_tiny_seed42
SKIP completed: cric_5pct_pseudo_label_convnext_tiny_seed42
SKIP completed: cric_5pct_mean_teacher_convnext_tiny_seed42
SKIP completed: cric_5pct_fixmatch_convnext_tiny_seed42
SKIP completed: cric_10pct_supervised_convnext_tiny_seed42
SKIP completed: cric_10pct_pseudo_label_convnext_tiny_seed42
SKIP completed: cric_10pct_mean_teacher_convnext_tiny_seed42
SKIP completed: cric_10pct_fixmatch_convnext_tiny_seed42
SKIP completed: riva_5pct_supervised_convnext_tiny_seed42
SKIP completed: riva_5pct_pseudo_label_convnext_tiny_seed42
SKIP completed: riva_5pct_mean_teacher_convnext_tiny_seed42
RESUME riva_5pct_fixmatch_convnext_tiny_seed42: epoch 26, batch 18
riva_5pct_fixmatch_convnext_tiny_seed42 | epoch 027/100 | loss=0.1083 | val_macro_f1=0.6665 | best=0.7313 | patience=2/12
riva_5pct_fixmatch_convnext_tiny_seed42 | epoch 028/100 | loss=0.0498 | val_macro_f1=0.6871 | best=0.7313 | patience=3/12
riva_5pct_fixmatch_convnext_tiny_seed42

## 7. Cross-dataset evaluation

After the source-domain runs finish, this cell evaluates each source-trained best checkpoint on the **other dataset's fixed test split**. No target test labels are used for training, checkpoint selection, or threshold tuning. The decision threshold remains 0.5. These results quantify transfer/generalization and should be reported separately from within-dataset test metrics.

In [7]:
def cross_dataset_evaluation():
    rows=[]
    for source in DATASETS:
        target = "riva" if source == "cric" else "cric"
        target_df = manifests[target].query("split == 'test'").reset_index(drop=True)
        target_dl = loader(target_df, "eval")
        for frac in FRACTIONS:
            for method in METHODS:
                name = run_name(source, frac, method)
                best = OUT_DIR / name / "checkpoint_best.pt"
                if not best.exists():
                    print("MISSING:", best)
                    continue
                model = build_model()
                payload = torch.load(best, map_location=DEVICE, weights_only=False)
                model.load_state_dict(payload["model"])
                m = evaluate(model, target_dl)
                rows.append({"source_dataset":source, "target_dataset":target,
                             "label_fraction":frac, "method":method, **m})
                del model
                if torch.cuda.is_available(): torch.cuda.empty_cache()
    if rows:
        out = pd.DataFrame(rows)
        out.to_csv(OUT_DIR / "cross_dataset_summary.csv", index=False)
        display(out.sort_values(["source_dataset","label_fraction","method"]))
        print("Saved:", OUT_DIR / "cross_dataset_summary.csv")
    else:
        print("No best checkpoints found.")
    return rows

# Run after the main queue has completed:
# cross_results = cross_dataset_evaluation()


## 8. Results table and comparison to supervised baselines

The source-test comparison below joins each semi-supervised method with the supervised result at the same dataset and label budget. It reports absolute metric differences, not percentage points mislabeled as percent. Use validation Macro-F1 for model selection and test results for final reporting only after the experiment matrix is frozen.

In [8]:
summary_file = OUT_DIR / "phase11_summary.csv"
if summary_file.exists():
    summary = pd.read_csv(summary_file)
    metrics = ["accuracy","precision","recall_sensitivity","specificity","macro_f1","auroc"]
    base = summary[summary.method == "supervised"][["dataset","label_fraction"] + metrics].copy()
    base = base.rename(columns={m:f"supervised_{m}" for m in metrics})
    comparison = summary.merge(base, on=["dataset","label_fraction"], how="left")
    for m in metrics:
        comparison[f"delta_{m}"] = comparison[m] - comparison[f"supervised_{m}"]
    comparison.to_csv(OUT_DIR / "phase11_comparison_vs_supervised.csv", index=False)
    display(comparison.sort_values(["dataset","label_fraction","method"]))
    print("Saved comparison:", OUT_DIR / "phase11_comparison_vs_supervised.csv")
else:
    print("Run the experiment queue first.")


,run_id,dataset,method,label_fraction,backbone,best_epoch,best_val_macro_f1,accuracy,precision,recall_sensitivity,...,supervised_recall_sensitivity,supervised_specificity,supervised_macro_f1,supervised_auroc,delta_accuracy,delta_precision,delta_recall_sensitivity,delta_specificity,delta_macro_f1,delta_auroc
0,cric_5pct_fixmatch_convnext_tiny_seed42,cric,fixmatch,0.05,convnext_tiny,33,0.852027,0.869961,0.761658,0.835227,...,0.754261,0.911416,0.836965,0.930047,0.006502,-0.027346,0.080966,-0.026201,0.013608,0.003249
1,cric_5pct_mean_teacher_convnext_tiny_seed42,cric,mean_teacher,0.05,convnext_tiny,10,0.824242,0.830949,0.711022,0.751420,...,0.754261,0.911416,0.836965,0.930047,-0.032510,-0.077983,-0.002841,-0.045540,-0.033225,-0.020890
2,cric_5pct_pseudo_label_convnext_tiny_seed42,cric,pseudo_label,0.05,convnext_tiny,15,0.827479,0.846121,0.726329,0.795455,...,0.754261,0.911416,0.836965,0.930047,-0.017339,-0.062675,0.041193,-0.043044,-0.013850,-0.011923
3,cric_5pct_supervised_convnext_tiny_seed42,cric,supervised,0.05,convnext_tiny,13,0.841075,0.863459,0.789004,0.754261,...,0.754261,0.911416,0.836965,0.930047,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,cric_10pct_fixmatch_convnext_tiny_seed42,cric,fixmatch,0.10,convnext_tiny,3,0.868045,0.871261,0.773890,0.816761,...,0.808239,0.911416,0.858825,0.941593,-0.008669,-0.026392,0.008523,-0.016220,-0.008341,-0.001753
5,cric_10pct_mean_teacher_convnext_tiny_seed42,cric,mean_teacher,0.10,convnext_tiny,50,0.878472,0.877330,0.773022,0.846591,...,0.808239,0.911416,0.858825,0.941593,-0.002601,-0.027259,0.038352,-0.020586,0.000164,0.004003
6,cric_10pct_pseudo_label_convnext_tiny_seed42,cric,pseudo_label,0.10,convnext_tiny,7,0.873140,0.859991,0.723329,0.876420,...,0.808239,0.911416,0.858825,0.941593,-0.019939,-0.076952,0.068182,-0.058640,-0.015380,-0.003769
7,cric_10pct_supervised_convnext_tiny_seed42,cric,supervised,0.10,convnext_tiny,10,0.877956,0.879931,0.800281,0.808239,...,0.808239,0.911416,0.858825,0.941593,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,riva_5pct_fixmatch_convnext_tiny_seed42,riva,fixmatch,0.05,convnext_tiny,24,0.731269,0.786921,0.696035,0.701599,...,0.609236,0.887923,0.758340,0.857992,-0.002816,-0.051241,0.092362,-0.054589,0.008638,0.006911
9,riva_5pct_mean_teacher_convnext_tiny_seed42,riva,mean_teacher,0.05,convnext_tiny,8,0.721238,0.784418,0.662454,0.791297,...,0.609236,0.887923,0.758340,0.857992,-0.005319,-0.084823,0.182060,-0.107246,0.014383,0.005746


Saved comparison: C:\Users\nanda\Documents\cervical-semi-sl\data\results\phase11_controlled_v1\phase11_comparison_vs_supervised.csv


## Output files

All outputs are stored under `data/results/phase11_controlled_v1/`:

- `label_subsets/`: deterministic 5% and 10% labeled-subset manifests.
- `<run_id>/checkpoint_latest.pt`: restart checkpoint, updated after each training batch.
- `<run_id>/checkpoint_best.pt`: weights selected by validation Macro-F1.
- `<run_id>/history.csv`: epoch-level training loss and validation metrics.
- `<run_id>/test_metrics.json`: final test metrics for that run.
- `<run_id>/DONE.json`: completion marker used to skip finished runs.
- `phase11_summary.csv`: rolling summary across completed runs.
- `phase11_comparison_vs_supervised.csv`: differences from the matching supervised baseline.
- `cross_dataset_summary.csv`: cross-dataset test results from Section 7.

## Reporting checklist

- Report CRIC and RIVA results separately for 5% and 10% labels.
- Include all listed metrics and confusion-matrix counts, especially abnormal-class sensitivity.
- Report cross-dataset results separately; do not call source-domain test scores “generalization”.
- State the fixed split protocol, seed, labeled fraction definition, backbone, augmentation, checkpoint-selection metric, and early-stopping policy.
- For stronger evidence, repeat the final shortlist with multiple seeds and report mean ± standard deviation.
- These outputs are research experiments, not a clinically validated diagnostic system.